# PCA on Drilling Data
## Machine learning for energy applications — Module 4, Exercise 1

In this exercise we apply **Principal Component Analysis (PCA)** to a drilling dataset.
PCA is one of the most widely used dimensionality-reduction techniques: it finds
orthogonal directions (principal components) that capture maximum variance in the data.

We will:
1. Explore and standardize the drilling features
2. Fit PCA and examine the explained variance
3. Visualize the data in the reduced PC space
4. Interpret the PCA loadings
5. Reconstruct the data and measure reconstruction error
6. Use PCA as preprocessing for a downstream clustering task

> **Exploratory analysis.** Every fit in this notebook (scaler, PCA, k-means) uses all 100 rows,
> because the question is what structure this one drilling table has. The results describe these
> rows; they are not a model for new data. If PCA fed a predictive model or monitored new depths,
> the scaler and PCA would be fitted on training rows only (for example inside a `Pipeline`), and
> the reconstruction error would be checked on held-out depths.

---
## 0 — Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Plotting defaults
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 12,
})

RANDOM_STATE = 42
print("Setup complete.")

---
## 1 — Load and Explore the Data

The dataset contains drilling measurements at various depths:
- **Depth** — measured depth along the wellbore
- **ROP** — rate of penetration
- **RPM** — rotational speed of the drill string
- **Torque** — torque applied to the drill string
- **Weight_on_Bit** — downward force on the drill bit

In [ ]:
df = pd.read_csv("drilling.csv")
print(f"Shape: {df.shape}")
df.head(10)

In [ ]:
df.describe()

In [ ]:
df.info()

### 1.1 Correlation Matrix Heatmap

PCA is most useful when features are correlated — it rotates the coordinate system
so that the first few components capture most of the variance.

In [ ]:
corr = df.corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

---
## 2 — Standardize Features

PCA is sensitive to scale — features with larger numeric ranges would dominate the
variance calculation. We standardize to zero mean and unit variance. The scaler is fitted on
all 100 rows because this is an exploratory description of this table.

We treat **Depth** as a positional index rather than a feature to reduce.

In [ ]:
feature_cols = ["ROP", "RPM", "Torque", "Weight_on_Bit"]
X = df[feature_cols].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature means after scaling : {X_scaled.mean(axis=0).round(6)}")
print(f"Feature stds  after scaling : {X_scaled.std(axis=0).round(6)}")

---
## 3 — Apply PCA (All Components)

We fit PCA retaining **all** components first so we can inspect how variance
is distributed across them.

In [ ]:
pca_full = PCA(n_components=len(feature_cols), random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)

print("Explained variance ratio per component:")
for i, var in enumerate(pca_full.explained_variance_ratio_):
    print(f"  PC{i+1}: {var:.4f} ({var*100:.2f}%)")

print(f"\nTotal explained variance: {pca_full.explained_variance_ratio_.sum():.4f}")

### 3.1 Scree Plot and Cumulative Variance

In [ ]:
explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)
components = np.arange(1, len(explained) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].bar(components, explained, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Principal Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Scree Plot")
axes[0].set_xticks(components)

# Cumulative variance
axes[1].plot(components, cumulative, "o-", color="tomato", linewidth=2)
axes[1].axhline(y=0.90, color="gray", linestyle="--", label="90% threshold")
axes[1].axhline(y=0.95, color="gray", linestyle=":", label="95% threshold")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Explained Variance")
axes[1].set_xticks(components)
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 4 — Choose Number of Components

We select the smallest number of components that capture at least **90-95%** of
the total variance. The threshold is a choice, not a rule: justify it from the purpose (for
example a compression tolerance for each channel), and read the scree plot as well.

In [ ]:
threshold = 0.95
n_components = np.argmax(cumulative >= threshold) + 1
print(f"Components needed for {threshold*100:.0f}% variance: {n_components}")
print(f"Cumulative variance with {n_components} components: {cumulative[n_components-1]:.4f}")

In [ ]:
pca_reduced = PCA(n_components=n_components, random_state=RANDOM_STATE)
X_pca_reduced = pca_reduced.fit_transform(X_scaled)

print(f"Original shape : {X_scaled.shape}")
print(f"Reduced shape  : {X_pca_reduced.shape}")
if n_components == len(feature_cols):
    print("No reduction: every component is needed to reach the threshold.")

---
## 5 — Visualize Data in PC1 vs PC2 Space

We project the data onto the first two principal components and colour by **Depth**
to see whether there are depth-related patterns in the drilling parameters.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
scatter = ax.scatter(X_pca_full[:, 0], X_pca_full[:, 1],
                     c=df["Depth"], cmap="viridis", alpha=0.7, s=30, edgecolors="w", linewidth=0.3)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("Depth")
ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}% variance)")
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}% variance)")
ax.set_title("Drilling Data in PC1-PC2 Space (coloured by Depth)")
plt.tight_layout()
plt.show()

---
## 6 — Analyse PCA Loadings

The **loadings** (rows of `pca.components_`) tell us how much each original feature
contributes to each principal component. Large absolute loadings indicate strong
contributions.

In [ ]:
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=[f"PC{i+1}" for i in range(len(feature_cols))],
    index=feature_cols
)
print("PCA Loadings (feature contributions to each PC):")
loadings.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(loadings, annot=True, fmt=".3f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax)
ax.set_title("PCA Loadings Heatmap")
ax.set_ylabel("Original Feature")
ax.set_xlabel("Principal Component")
plt.tight_layout()
plt.show()

### 6.1 Biplot — Loadings on the PC1-PC2 Plane

A biplot overlays the loading vectors on the scatter plot so we can see which
features drive the spread in PC space.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(X_pca_full[:, 0], X_pca_full[:, 1], alpha=0.3, s=15, color="gray")

# Overlay loading vectors (scaled for visibility)
scale = 3.0
for i, feat in enumerate(feature_cols):
    ax.arrow(0, 0,
             pca_full.components_[0, i] * scale,
             pca_full.components_[1, i] * scale,
             head_width=0.08, head_length=0.05, fc="red", ec="red")
    ax.text(pca_full.components_[0, i] * scale * 1.15,
            pca_full.components_[1, i] * scale * 1.15,
            feat, fontsize=11, color="red", fontweight="bold")

ax.set_xlabel(f"PC1 ({explained[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({explained[1]*100:.1f}%)")
ax.set_title("Biplot — PCA Loadings on PC1-PC2")
plt.tight_layout()
plt.show()

---
## 7 — Reconstruction from Reduced Dimensions

When we reduce dimensions we lose some information. We can reconstruct an
approximation of the original data using `inverse_transform` and measure how
much error the compression introduces.

Report the error **per feature, in original units** (the CSV does not state units, so use the
file's own). A single error averaged over features with different units would be dominated by
the feature with the largest numbers (Weight_on_Bit). If the chosen number of components equals
the number of features, the reconstruction is exact, so we also compare 1 to 4 components.
These are in-sample errors on the same 100 rows used to fit the scaler and PCA; a compression
decision for new data would check them on held-out depths.

In [ ]:
# Reconstruct using the reduced-dimension PCA (the n_components chosen above)
X_reconstructed_scaled = pca_reduced.inverse_transform(X_pca_reduced)

# Undo the scaling to get back to original units
X_reconstructed = scaler.inverse_transform(X_reconstructed_scaled)

# RMSE per feature in original units (do not average across features with different units)
rmse_per_feature = np.sqrt(np.mean((X - X_reconstructed) ** 2, axis=0))
print(f"Reconstruction RMSE per feature with {n_components} components (original units):")
for feat, err in zip(feature_cols, rmse_per_feature):
    print(f"  {feat:15s}: {err:.3f}")

In [ ]:
# Compare reconstruction error for different numbers of components
errors = []       # mean squared error in standardized units (every feature weighted equally)
rmse_by_k = {}    # RMSE per feature in original units
for nc in range(1, len(feature_cols) + 1):
    pca_tmp = PCA(n_components=nc, random_state=RANDOM_STATE)
    X_tmp = pca_tmp.fit_transform(X_scaled)
    X_rec = pca_tmp.inverse_transform(X_tmp)
    errors.append(np.mean((X_scaled - X_rec) ** 2))
    rmse_by_k[nc] = np.sqrt(np.mean((X - scaler.inverse_transform(X_rec)) ** 2, axis=0))

rmse_table = pd.DataFrame(rmse_by_k, index=feature_cols).T
rmse_table.index.name = "components kept"
print("Reconstruction RMSE per feature, original units:")
print(rmse_table.round(2))
print("\nStandard deviation of each feature, for scale:")
print(df[feature_cols].std().round(2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(feature_cols) + 1), errors, "o-", color="steelblue", linewidth=2)
ax.set_xlabel("Number of Components")
ax.set_ylabel("Mean Reconstruction Error (MSE, standardized)")
ax.set_title("Reconstruction Error vs Number of PCA Components")
ax.set_xticks(range(1, len(feature_cols) + 1))
plt.tight_layout()
plt.show()

---
## 8 — PCA as Preprocessing for Clustering

Dimensionality reduction can help clustering by removing noise dimensions. We compare K-Means
clustering on the standardized features with K-Means on the PCA-reduced data (exploratory: all
fits use the same 100 rows).

Two cautions when reading the result:
- If `n_components` equals the number of features, PCA is only a rotation, so K-Means finds
  the same clusters and the same silhouette score.
- Silhouette scores computed in spaces of different dimension are not directly comparable.
  For a fair comparison, also score the PCA-space clusters in the standardized feature space.

In [ ]:
n_clusters = 3

# Clustering on the standardized features
km_orig = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
labels_orig = km_orig.fit_predict(X_scaled)
sil_orig = silhouette_score(X_scaled, labels_orig)

# Clustering on PCA-reduced data
km_pca = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
labels_pca = km_pca.fit_predict(X_pca_reduced)
sil_pca = silhouette_score(X_pca_reduced, labels_pca)

# Fair comparison: the PCA-space clusters scored in the same standardized feature space
sil_pca_in_orig = silhouette_score(X_scaled, labels_pca)

print(f"Silhouette score (original {X_scaled.shape[1]}D)   : {sil_orig:.4f}")
print(f"Silhouette score (PCA-reduced {X_pca_reduced.shape[1]}D): {sil_pca:.4f}")
print(f"PCA-space clusters scored in the original {X_scaled.shape[1]}D space: {sil_pca_in_orig:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Project original-space clusters into PC space for visualization
axes[0].scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=labels_orig,
                cmap="Set1", alpha=0.6, s=25, edgecolors="w", linewidth=0.3)
axes[0].set_title(f"KMeans on Original Data\nSilhouette = {sil_orig:.3f}")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(X_pca_full[:, 0], X_pca_full[:, 1], c=labels_pca,
                cmap="Set1", alpha=0.6, s=25, edgecolors="w", linewidth=0.3)
axes[1].set_title(f"KMeans on PCA-Reduced Data\nSilhouette = {sil_pca:.3f}")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.suptitle("Clustering Comparison: Original vs PCA-Reduced", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9 — Extension (optional): PCA on Raw Units

Repeat PCA without standardizing and compare PC1 with the standardized fit. On raw units the
variance is dominated by the feature with the largest numeric spread, so PC1 then describes the
units of measurement rather than drilling behaviour.

In [ ]:
# PCA on the raw (unscaled) features, for comparison only
pca_raw = PCA(n_components=len(feature_cols), random_state=RANDOM_STATE)
pca_raw.fit(X)
print("Explained variance ratio, raw units    :", pca_raw.explained_variance_ratio_.round(4))
print("Explained variance ratio, standardized :", pca_full.explained_variance_ratio_.round(4))

pc1_comparison = pd.DataFrame({
    "PC1 loading, raw units": pca_raw.components_[0],
    "PC1 loading, standardized": pca_full.components_[0],
    "Std of feature (raw units)": X.std(axis=0, ddof=1),
}, index=feature_cols)
pc1_comparison.round(4)

---
## Summary

In this exercise we applied PCA to drilling data and covered:

1. **Data exploration** — correlation structure among drilling parameters.
2. **Standardization** — essential preprocessing for PCA since it is scale-sensitive.
3. **Explained variance** — scree plots and cumulative variance to choose the number of components.
4. **Visualization** — projecting data into PC1-PC2 space and colouring by Depth.
5. **Loadings analysis** — understanding which features drive each principal component (biplot).
6. **Reconstruction error** — per feature, in original units, for each number of components.
7. **PCA + Clustering** — using PCA as a preprocessing step for K-Means, compared in the same space.
8. **Raw units (extension)** — why PC1 of unscaled data follows the largest numeric spread.

**Key takeaway**: PCA is linear, fast, and interpretable, and it compresses well when channels are
correlated. Whether it compresses a particular table is an empirical question: read the spectrum,
the loadings, and the reconstruction error per channel in original units. A flat spectrum means
the channels carry separate information, which is a finding, not a failure. The fits here describe
these 100 rows (exploratory); for prediction or monitoring, fit the scaler and PCA on training
data only.